## Import

In [1]:
import os
import gc
import math
import pickle
from tqdm import tqdm

import warnings
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torchvision.transforms as T
from box import Box
from timm import create_model
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 

import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

print(pl.__version__)
warnings.filterwarnings("ignore")

## Config

In [2]:
config = {'exp_name':'baseline_v1',
          'seed': 2023,
          'root': '../input/petfinder-pawpularity-score/', 
          'n_splits': 5,
          'n_epochs': 5,
          'early_stop': 5,
          'lr': 1e-4,
          'pretrain': True,
          'image_size': 384,
          'train_loader': {
              'batch_size': 16,
              'shuffle': True,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'val_loader': {
              'batch_size': 32,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'model':{
              'name': 'swin_large_patch4_window12_384',
              'output_dim': 1
          },
          'loss': 'nn.CrossEntropyLoss',
}

config = Box(config)

## Fix Seed

In [3]:
seed_everything(config.seed)

Global seed set to 2023


2023

## Tools

In [4]:
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    assert alpha > 0, "alpha should be larger than 0"
    assert x.size(0) > 1, "Mixup cannot be applied to a single instance."

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0])
    mixed_x = lam * x + (1 - lam) * x[rand_index, :]
    target_a, target_b = y, y[rand_index]
    return mixed_x, target_a, target_b, lam





def compute_accuracy(y, target):
    """Compute the accuracy
    Args:
    ----------
        y: the output of model
        target: the ground truth of model predict

    Returns:
    ----------
        The accuracy of model prediction
    """
    pred = torch.max(y, 1)[1]
    correct = (pred == target)
    return (sum(correct) / len(correct)).detach().cpu().item()

## Dataset

In [5]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None, image_size=224):
        
        self._X = df["Id"].values
        self._y = None
        self.transform = transform
        
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            self._y = df["Pawpularity"].values
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        image_path = self._X[idx]
        image = read_image(image_path)
        image = self.transform(image)
        
        if self._y is not None:
            label = self._y[idx]
            return image, label
        return image

## Model

In [6]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        self._criterion = eval(self.hparams.loss)()
        self.metrics = compute_accuracy
        
        self.validation_step_outputs = {'val/logits': [],
                                        'val/pred': [],
                                        'val/labels': []}  # 用來計算epoch的val/loss, val/rmse
        
        self.__build_model()
        
        
    def __build_model(self):
        self.backbone = create_model(self.hparams.model.name,
                                     pretrained=self.hparams.pretrain, 
                                     num_classes=0, 
                                     in_chans=3)
        num_features = self.backbone.num_features
        self.fc = nn.Sequential(nn.Dropout(0.5), 
                                nn.Linear(num_features, 
                                          self.hparams.model.output_dim))

    def forward(self, x):
        f = self.backbone(x)
        out = self.fc(f)
        return out

    def training_step(self, batch, batch_idx):
        images, labels = batch
        
        # Mixup (50%的機率)
        if torch.rand(1)[0] < 0.5:
            mix_images, target_a, target_b, lam = mixup(images, labels, alpha=0.5)
            logits = self(mix_images)
            loss = self._criterion(logits, target_a) * lam + (1 - lam) * self._criterion(logits, target_b)
            
        else:
            logits = self(images)
            loss = self._criterion(logits, labels)
            
        # # 顯示predict結果
        # pred = logits.sigmoid() * 100.
        # labels = labels * 100.
        # print(f"pred: {pred}")
        # print(f"labels: {labels}")
            
        self.log("train/loss", loss, prog_bar=True)
        return loss
        
    def validation_step(self, batch, batch_idx):
        images, labels = batch
        
        logits = self(images)
        loss = self._criterion(logits, labels)
        
        self.validation_step_outputs['val/logits'].append(logits)
        self.validation_step_outputs['val/labels'].append(labels)
        
        # return {'val/pred': pred, 'val/labels': labels, 'val/loss': loss}
        return {'val/loss': loss}
    
    def on_validation_epoch_end(self):
        logits = torch.cat(self.validation_step_outputs['val/logits'], dim=0)
        labels = torch.cat(self.validation_step_outputs['val/labels'], dim=0)
        pred = torch.sigmoid(logits)
        
        # # 顯示predict結果
        # pred = logits.sigmoid() * 100.
        # labels = labels * 100.
        # print(f"pred: {pred}")
        # print(f"labels: {labels}")

        loss = self._criterion(logits, labels)
        metric = self.metrics(pred, labels)
        
        self.log('val/loss', loss, prog_bar=True)
        self.log('val/metric', metric, prog_bar=True)
        
        self.validation_step_outputs['val/logits'].clear()
        self.validation_step_outputs['val/pred'].clear()
        self.validation_step_outputs['val/labels'].clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

## Test

In [27]:
def rmse(predict,target):
    return 100. * ((predict - target) ** 2).mean() ** 0.5

device = 'cuda'

warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

train_transform = T.Compose([T.Resize(config.image_size),
                                T.CenterCrop([config.image_size, config.image_size]),
                                T.RandomHorizontalFlip(),
                                T.RandomVerticalFlip(),
                                T.RandomAffine(15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
                                T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                                T.ConvertImageDtype(torch.float),
                                T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

val_transform = T.Compose([T.Resize(config.image_size),
                            T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

stage = 'train'
df = pd.read_csv(os.path.join(config.root, stage+'.csv'))
df["Id"] = df["Id"].apply(lambda x: os.path.join(config.root, stage, x + ".jpg")) # 將ID改成圖片路徑
df["Pawpularity"] = df["Pawpularity"].astype(float).apply(lambda x: x / 100.) # 將Pawpularity軟換到[0, 1]

# df to dataset
val_data = PetfinderDataset(df, val_transform, config.image_size)
val_loader = DataLoader(val_data, **config.val_loader)

fold_predicts = []
# labels = []
for fold in range(config.n_splits):
    model = Model(config).load_from_checkpoint(f'baseline_v1/fold_{fold}/version_0/best.ckpt')
    
    model.to(device)
    model.eval()
    
    predicts = []
    pbar = tqdm(val_loader)
    pbar.set_description(f'Fold {fold}')

    for images, target in pbar:
        images = images.to(device)
        with torch.no_grad():
            predict = model(images).sigmoid().detach().cpu()
            
        predicts.append(predict.flatten().numpy())
        # if fold == 0:
        #     labels.append(target.numpy())

    fold_predicts.append(np.concatenate(predicts).flatten())
    
    del model
    torch.cuda.empty_cache()
    gc.collect()

mean_predicts = np.array(fold_predicts).mean(axis=0)
# labels = np.concatenate(np.array(labels)).flatten()

# 計算RMSE
metric = rmse(mean_predicts, df["Pawpularity"].to_numpy())
print(f"metric: {metric}")

# 儲存csv
mean_predicts_df = pd.DataFrame(mean_predicts, columns=["mean_predicts"])
df = pd.concat([df, mean_predicts_df], axis=1)
df.to_csv("predicts.csv", index=False)

# 存成pickle
with open('predicts.pickle', 'wb') as f:
    pickle.dump(mean_predicts, f)

Fold 4: 100%|██████████| 310/310 [01:55<00:00,  2.67it/s]
